# Demo 4: Semantic Correspondence — DINOv2

**Model:** `facebook/dinov2-base`  
**Task:** Click a point on the Source image → find the semantically corresponding location in the Target image via a heatmap

---
**Flow:**
1. Load Source / Target images (webcam or file)
2. Load DINOv2 model
3. Click on Source image → extract patch feature
4. Compute cosine similarity with all Target patches → visualize heatmap
5. Multi-point correspondence


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import cv2

## Step 1: Load Source / Target Images

- **Source** — the image containing the object you want to query (click this image)
- **Target** — the image where you want to find the corresponding point

Works best when the same person or object appears in both images.


In [ ]:
# ── Load from file ───────────────────────────────────────────────────────────
SOURCE_PATH = "sample.png"   # image to click on
TARGET_PATH = "sample.png"   # image to search in (replace with a different image)

source_image = Image.open(SOURCE_PATH).convert("RGB")
target_image = Image.open(TARGET_PATH).convert("RGB")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(source_image); axes[0].set_title("Source Image"); axes[0].axis('off')
axes[1].imshow(target_image); axes[1].set_title("Target Image"); axes[1].axis('off')
plt.tight_layout()
plt.show()
print(f"Source: {source_image.width} x {source_image.height}")
print(f"Target: {target_image.width} x {target_image.height}")

## Step 2: Load DINOv2 Model

> `facebook/dinov2-base` — 86M params, patch size 14×14.  
> GPU loading takes about 5–10 seconds.


In [ ]:
from models.dino_correspondence import DINOCorrespondence

corr_model = DINOCorrespondence()
corr_model.load()

## Step 3: Click Points on the Source Image

- **Left click** → add query point (colored red/blue/green in order)
- **Undo button** → remove last point
- **Clear button** → reset all points

Once done, run **Step 4**.


In [ ]:
import ipympl  # noqa — registers 'widget' backend
import ipywidgets as widgets
from IPython.display import display
%matplotlib widget

from utils.correspondence_vis import POINT_COLORS_RGB

QUERY_POINTS = []   # [(x, y), ...]
point_artists = []

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(source_image)
ax.set_title("Source Image — click to add query points", fontsize=12)
ax.axis('off')
plt.tight_layout()

def redraw_points():
    for a in point_artists:
        a.remove()
    point_artists.clear()
    for i, (px, py) in enumerate(QUERY_POINTS):
        c = [v/255 for v in POINT_COLORS_RGB[i % len(POINT_COLORS_RGB)]]
        art = ax.plot(px, py, 'o', color=c, markersize=14,
                      markeredgecolor='white', markeredgewidth=2)[0]
        txt = ax.text(px + 12, py - 12, str(i+1), color=c, fontsize=11, fontweight='bold')
        point_artists.extend([art, txt])
    fig.canvas.draw_idle()

def on_click(event):
    if event.inaxes != ax or event.xdata is None:
        return
    x, y = int(event.xdata), int(event.ydata)
    QUERY_POINTS.append((x, y))
    redraw_points()
    print(f"[Point {len(QUERY_POINTS)}] ({x}, {y})  |  QUERY_POINTS = {QUERY_POINTS}")

fig.canvas.mpl_connect('button_press_event', on_click)

btn_undo  = widgets.Button(description='Undo',  button_style='warning', layout=widgets.Layout(width='100px'))
btn_clear = widgets.Button(description='Clear', button_style='danger',  layout=widgets.Layout(width='100px'))

def on_undo(_):
    if QUERY_POINTS:
        removed = QUERY_POINTS.pop()
        redraw_points()
        print(f"Undo: {removed} removed  — {len(QUERY_POINTS)} point(s) remaining")

def on_clear(_):
    QUERY_POINTS.clear()
    redraw_points()
    print("All points cleared.")

btn_undo.on_click(on_undo)
btn_clear.on_click(on_clear)
display(widgets.HBox([btn_undo, btn_clear]))
plt.show()

## Step 4: Compute Correspondence & Visualize Heatmap

For each clicked point, finds the most similar location in the Target image.


In [ ]:
%matplotlib inline

from utils.correspondence_vis import apply_heatmap, draw_correspondence_points, make_heatmap_result

if not QUERY_POINTS:
    print("No points selected. Click on the image in Step 3.")
else:
    heatmaps, best_matches = corr_model.find_multi_correspondence(
        source_image, QUERY_POINTS, target_image
    )

    n = len(QUERY_POINTS)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n),
                              squeeze=False)

    for i, ((qx, qy), hmap, (bx, by)) in enumerate(zip(QUERY_POINTS, heatmaps, best_matches)):
        color_f = [v/255 for v in POINT_COLORS_RGB[i % len(POINT_COLORS_RGB)]]

        # Left: Source with query point
        axes[i][0].imshow(source_image)
        axes[i][0].plot(qx, qy, 'o', color=color_f, markersize=14,
                        markeredgecolor='white', markeredgewidth=2)
        axes[i][0].set_title(f"Source — Point {i+1} ({qx},{qy})"); axes[i][0].axis('off')

        # Center: Heatmap only
        im = axes[i][1].imshow(hmap, cmap='jet', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[i][1], fraction=0.046, pad=0.04)
        axes[i][1].set_title(f"Similarity Heatmap — Point {i+1}"); axes[i][1].axis('off')

        # Right: Target overlay + best match
        target_vis = make_heatmap_result(target_image, hmap, (bx, by))
        axes[i][2].imshow(target_vis)
        axes[i][2].set_title(f"Target — Best Match ({bx},{by})"); axes[i][2].axis('off')

        print(f"Point {i+1}: Source ({qx},{qy}) → Target ({bx},{by})")

    plt.suptitle("DINOv2 Semantic Correspondence", fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Step 5: Multi-point Comparison

Show all source/target point pairs with matching colors on both images.


In [ ]:
%matplotlib inline

if not QUERY_POINTS:
    print("No points selected.")
else:
    heatmaps, best_matches = corr_model.find_multi_correspondence(
        source_image, QUERY_POINTS, target_image
    )

    # Use the last point's heatmap as background
    tgt_overlay = make_heatmap_result(target_image, heatmaps[-1], best_matches[-1])

    src_ann, tgt_ann = draw_correspondence_points(
        source_image, QUERY_POINTS, tgt_overlay, best_matches
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(src_ann); axes[0].set_title("Source (Query Points)"); axes[0].axis('off')
    axes[1].imshow(tgt_ann); axes[1].set_title("Target (Best Matches)"); axes[1].axis('off')

    # Legend
    patches = [
        mpatches.Patch(color=[v/255 for v in POINT_COLORS_RGB[i % len(POINT_COLORS_RGB)]],
                       label=f"Point {i+1}: ({qx},{qy})→({bx},{by})")
        for i, ((qx,qy),(bx,by)) in enumerate(zip(QUERY_POINTS, best_matches))
    ]
    axes[1].legend(handles=patches, loc='lower right', fontsize=9)

    plt.suptitle("DINOv2 Multi-point Semantic Correspondence", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()